# MirrorTopology Step 1 — Phase D-3b：保存 bank の実行後検証（read-only；family×size の partition 単位；v0.1）
Drive 上の 1 partition の出力（`<FAMILY>_<SIZE>_<run id>/out`；追加 9 配置・27 unit・NPZ 45）を，受入れ済み commit の engine で**読取り専用**に検証する：D-3b 生成 ledger（`registered_assets/d3b/d3b_generation_ledger.json`：9 run の固定 record／unit SHA／期待 NPZ SHA；pins と再導出に束縛）への accepted-run binding，検証 source の inventory 束縛（生成 source と別記録），出力先の隔離，全 unit の `verify_twelve_bank_dir`（context への束縛・bytes・sidecar⇄manifest・member 集合・dtype・有限性・cid・範囲・UID・key・**全配列の SHA**），NPZ file SHA の再計算と ledger／final record の照合，配列統計，partition 内 9 配置の cid 同一（同 family latent），任意で D-2 family reference（`D2_REF_ROOT`）との cid 同一，N₀＋3N₀ 構造。Drive は変更しない。生成・較正・判定は行わない。`all_ok` は file/member/array integrity であり，科学的 support/strong ではない。出力は小さな JSON（`d3b_verify_<FAMILY>_<SIZE>.json`）と zip。**9 partition を 1 つずつ別 run で**（1 partition ≈10〜20 min の見込み）。

In [ ]:
# --- 0. OUTER LOCK (the only editable cell)
REPO_URL = 'https://github.com/tsujikeita/mirror-topology.git'
REPO_COMMIT = '<full 40-hex commit of the verification target (must contain d/d3b_verify_banks.py and registered_assets/d3b/)>'
EXPECTED_INVENTORY_SHA256 = '<sha256 of engine/phaseB/B2_completion_inventory.json inside that commit>'
FAMILY = 'E2'
SIZE = 'L1.00'
RUN_ROOT = '/content/drive/MyDrive/MirrorTopology_D3b/E2_L1.00_20260930T010656Z/out'    # the partition run's out/ (contains d3b/ and d3b_final_record.json)
D2_REF_ROOT = ''                   # optional: the D-2 family run's out/d2 (holds ref_<FAMILY>_b0 / _b1 / _fit) for the cid correspondence with the fixed reference; '' to skip


In [ ]:
# --- 1. fresh scratch checkout; inventory + script + ledger bound; Drive mounted (read-only usage)
import subprocess, sys, os, json, hashlib, shutil, time, re
sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT) and re.fullmatch(r'[0-9a-f]{64}', EXPECTED_INVENTORY_SHA256) and FAMILY in ('E2','E7','E8') and SIZE in ('L1.00','L1.20','L1.50')
from google.colab import drive; drive.mount('/content/drive'); assert os.path.exists(f'{RUN_ROOT}/d3b/d3_bank_registry.json'), RUN_ROOT
if D2_REF_ROOT: assert all(os.path.exists(f'{D2_REF_ROOT}/ref_{FAMILY}_{u}/COMPLETE.json') for u in ('b0','b1','fit')), D2_REF_ROOT
SCRATCH='/content/d3bv_scratch'; shutil.rmtree(SCRATCH, ignore_errors=True); subprocess.run(['git','clone','-q',REPO_URL,SCRATCH],check=True); subprocess.run(['git','-C',SCRATCH,'checkout','-q',REPO_COMMIT],check=True)
assert subprocess.check_output(['git','-C',SCRATCH,'rev-parse','HEAD']).decode().strip()==REPO_COMMIT and subprocess.check_output(['git','-C',SCRATCH,'status','--porcelain']).decode().strip()==''
PHASEB=f'{SCRATCH}/engine/phaseB'; INV=f'{PHASEB}/B2_completion_inventory.json'; assert sha(INV)==EXPECTED_INVENTORY_SHA256; inv=json.load(open(INV)); SCRIPT=f'{PHASEB}/d/d3b_verify_banks.py'; assert sha(SCRIPT)==inv['d_sha256']['d/d3b_verify_banks.py']
LEDGER=f'{PHASEB}/registered_assets/d3b/d3b_generation_ledger.json'; assert sha(LEDGER)==inv['registered_assets_sha256']['registered_assets/d3b/d3b_generation_ledger.json']; ledger=json.load(open(LEDGER)); part=ledger['partitions'][f'{FAMILY}_{SIZE}']; assert RUN_ROOT.rstrip('/').endswith(part['run_dir'].split('/')[-1]+'/out'), (RUN_ROOT, part['run_dir'])
pins=json.load(open(f'{PHASEB}/d/d3_pins.json')); ex=pins['environment']; subprocess.run([sys.executable,'-m','pip','install','-q',f"numpy=={ex['numpy']}",f"scipy=={ex['scipy']}",f"healpy=={ex['healpy']}",f"pot=={ex['pot']}",'threadpoolctl'],check=True)
OUT=f'/content/d3b_verify_{FAMILY}_{SIZE}'; shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT); lock=dict(commit=REPO_COMMIT, inventory_sha256=sha(INV), script_sha256=sha(SCRIPT), ledger_sha256=ledger['ledger_sha256'], family=FAMILY, size=SIZE, run_root=RUN_ROOT, run_id=part['run_id'], d2_ref_root=D2_REF_ROOT or None, engine=inv['engine_version']); json.dump(lock, open(f'{OUT}/verify_lock.json','w'), indent=1); print(lock)


In [ ]:
# --- 2. read-only verification (reads every NPZ twice: full array verification through verify_twelve_bank_dir, then statistics; the author estimate of ~10-20 min per partition is not a guarantee)
assert not os.path.exists(f'{OUT}/report'); rc=subprocess.run([sys.executable,SCRIPT,'--phaseb',PHASEB,'--run-root',RUN_ROOT,'--out',f'{OUT}/report','--mode','accepted']+(['--d2-ref-root',D2_REF_ROOT] if D2_REF_ROOT else []),capture_output=True,text=True)
# accepted mode; preserve stdout/stderr before opening the report
open(f'{OUT}/verify_stdout.txt','w').write(rc.stdout); open(f'{OUT}/verify_stderr.txt','w').write(rc.stderr); print(rc.stdout[-2500:]); print('rc', rc.returncode)
vp=f'{OUT}/report/d3b_verify_{FAMILY}_{SIZE}.json'
try:
    v=json.load(open(vp))
    if not isinstance(v, dict) or not isinstance(v.get('all_ok'), bool): raise ValueError('report is not a result dictionary with boolean all_ok')
except Exception as read_error:
    v=dict(stage='REPORT_MISSING_OR_INVALID', all_ok=False, coverage_complete=False, accepted_run_binding=False, launcher_fallback=True, failures=[f'verifier report unreadable: {read_error!r}', f'verifier returncode {rc.returncode}'], stderr_tail=rc.stderr[-2000:])
print('rc', rc.returncode, '| accepted_run_binding', v.get('accepted_run_binding'), '| scope', v.get('verification_scope'), '| all_ok', v.get('all_ok'), '| coverage_complete', v.get('coverage_complete'), '| cid', v.get('cid_correspondence_ok'), '| failures', v.get('failures'), '| seconds', round(v.get('seconds', 0)))
final_ok=bool(rc.returncode==0 and v.get('all_ok') is True and v.get('coverage_complete') is True and v.get('accepted_run_binding') is True and not v.get('launcher_fallback'))
json.dump(dict(lock=lock, exit_code=rc.returncode, stage=v.get('stage'), all_ok=v.get('all_ok'), coverage_complete=v.get('coverage_complete'), accepted_run_binding=v.get('accepted_run_binding'), launcher_fallback=bool(v.get('launcher_fallback')), verify_pass=final_ok), open(f'{OUT}/verify_final_record.json','w'), indent=1); print('verify_pass', final_ok)


In [ ]:
# --- 3. zip the evidence (small) for the audit
zp=f'/content/d3b_verify_{FAMILY}_{SIZE}_{REPO_COMMIT[:12]}.zip'; shutil.make_archive(zp[:-4], 'zip', root_dir=OUT)
from google.colab import files; print(zp, os.path.getsize(zp)); files.download(zp)
